# MedQA Model Fine-Tuning on Colab Pro

Fine-tune Qwen2.5-7B-Instruct on medical QA data using QLoRA.

**Requirements:**
- Colab Pro or Pro+ (for longer runtime)
- GPU runtime (T4 or A100)
- HuggingFace account (free)

**Runtime:** ~2-3 hours for 3 epochs on T4

**Output:** Fine-tuned LoRA adapters → merged model → uploaded to HuggingFace Hub

## Step 1: Setup & Install Dependencies

In [ ]:
# Install training dependencies
!pip install -q transformers peft bitsandbytes trl datasets accelerate huggingface-hub

# Clone portfolio repo (contains training scripts)
!git clone https://github.com/soham10i/portfolio.git

# Mount Google Drive to save models
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Prepare Training Data

In [ ]:
import json
import os

# Load your seed data from portfolio repo
with open('/content/portfolio/backend/data/medqa-seed.json') as f:
    data = json.load(f)

records = data if isinstance(data, list) else data.get('records', [])
print(f"Loaded {len(records)} seed examples")

# Optional: Download full MedQA dataset for better results
# !wget -q https://raw.githubusercontent.com/jind11/MedQA/master/data_clean/questions/US/4_options/phrases_no_exclude_train.jsonl -O /content/medqa-full.jsonl

# Convert to training format
output_path = '/content/medqa-train.jsonl'
with open(output_path, 'w') as f:
    for item in records:
        options = item.get('options', {})
        if isinstance(options, list):
            options = {chr(65+i): opt for i, opt in enumerate(options)}
        
        out = {
            'question': item.get('question', ''),
            'options': options,
            'answer': item.get('answer', ''),
            'explanation': item.get('explanation', '')
        }
        f.write(json.dumps(out, ensure_ascii=False) + '
')

print(f"Saved training data to {output_path}")

## Step 3: Configure Training

In [ ]:
# Training configuration
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # 7B model fits on T4
# MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"  # 14B needs A100 or longer training

DATASET_PATH = '/content/medqa-train.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/medqa-checkpoints/qwen-7b-medqa'

NUM_EPOCHS = 3
BATCH_SIZE = 4  # Reduce to 2 if OOM on T4
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")

## Step 4: Run QLoRA Training

In [ ]:
%cd /content/portfolio/training/medqa

!python train.py \
  --model_name {MODEL_NAME} \
  --dataset {DATASET_PATH} \
  --output_dir {OUTPUT_DIR} \
  --num_epochs {NUM_EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --learning_rate {LEARNING_RATE} \
  --lora_r {LORA_R} \
  --lora_alpha {LORA_ALPHA} \
  --save_steps 500 \
  --logging_steps 50

## Step 5: Merge LoRA Adapters into Base Model

In [ ]:
MERGED_DIR = '/content/drive/MyDrive/medqa-models/qwen-7b-medqa-merged'

!python scripts/merge_lora.py \
  --base_model {MODEL_NAME} \
  --adapter {OUTPUT_DIR} \
  --output {MERGED_DIR}

print(f"Merged model saved to: {MERGED_DIR}")

## Step 6: Test the Fine-Tuned Model

In [ ]:
!python scripts/test_model.py \
  --model {MERGED_DIR} \
  --question "A 45-year-old man has chest pain. What is the first line treatment?" \
  --options "Aspirin,Morphine,Nitroglycerin,Oxygen"

## Step 7: Upload to HuggingFace Hub

Get your token from: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import HfApi, login

# Login (you'll be prompted for token)
login()

HF_USERNAME = input("Your HuggingFace username: ")
REPO_ID = f"{HF_USERNAME}/qwen-medqa-finetuned"

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id=REPO_ID,
    commit_message="Upload fine-tuned MedQA model"
)

print(f"✅ Uploaded to https://huggingface.co/{REPO_ID}")

## Step 8: BLIP Fine-Tuning (Optional)

Only if you have domain-specific images for scene understanding.

In [ ]:
# Uncomment and modify if you have image-caption pairs
# %cd /content/portfolio/training/scene
# !pip install -r requirements.txt
# !python train_blip.py \
#   --image_dir /content/drive/MyDrive/scene-images \
#   --captions_file /content/drive/MyDrive/scene-captions.json \
#   --output_dir /content/drive/MyDrive/blip-finetuned \
#   --num_epochs 5

## Next Steps

1. **Cancel Colab Pro** if you only needed it for this training
2. **Deploy on RunPod Serverless** using the uploaded HuggingFace model
3. **Update Render env vars** to point to your RunPod endpoint

See `training/COST_OPTIMIZED_STRATEGY.md` in the repo for deployment steps.